# 实际收益率 Qlib 日历边界修复 v2

按冻结 Stage1 配套 Qlib 日历保留第一个训练日未知；零新行情请求、零收益读取、零拟合、零账户。

In [1]:
import csv,hashlib,json,os,tempfile
from datetime import datetime
from pathlib import Path
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence

root=Path.cwd();card=root/'research/stage1_treasury_real_yield_source_20261023'
repair=json.loads((card/'map_repair_plan_v2.json').read_text());source=json.loads((card/'source_receipt.json').read_text());binding=json.loads((card/'nas_binding.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert repair['frozen'] and not (card/'map_repair_v2_state.json').exists()
assert source['plan_sha256']==repair['parent_plan_sha256'] and binding['verified'] and binding['source_id']==repair['source_NAS_id']
for obj in binding['objects']:
    assert sha(root/obj['local_path'])==obj['sha256'] and (root/obj['local_path']).stat().st_size==obj['bytes']
raw=root/'data/snapshots/stage1_treasury_real_yield_source_20261023';rows=[]
for receipt in source['request_receipts']:
    p=raw/receipt['name'];assert sha(p)==receipt['raw_sha256']
    with p.open(newline='',encoding='utf-8-sig') as handle:
        reader=csv.DictReader(handle);columns=reader.fieldnames;record=list(reader)
    dcol=next(x for x in columns if x.strip().lower()=='date');vcol=next(x for x in columns if x.strip().lower()=='10 yr')
    rows.extend((pd.Timestamp(datetime.strptime(r[dcol],'%m/%d/%Y').date()),float(r[vcol])) for r in record)
rates=pd.Series(dict(rows)).sort_index();assert len(rates)==len(rows)==5956
state={'status':'started','plan_sha256':sha(card/'map_repair_plan_v2.json'),'new_external_GETs':0,'SOXX_outcome_reads':0,'new_model_fits':0,'new_native_accounts':0}
(card/'map_repair_v2_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
provider=root/'data/qlib/full_path_tiered_allocation_20261013'
experiment='stage1_treasury_real_yield_source_20261023_map_repair_v2'
qlib.init(provider_uri=str(provider),region='us',exp_manager={'class':'MLflowExpManager','module_path':'qlib.workflow.expm','kwargs':{'uri':str(card/'qlib_recorder'),'default_exp_name':experiment}})
cal=pd.DatetimeIndex(D.calendar(start_time='2018-10-01',end_time='2026-10-07',freq='day'))
records=[]
for i,day in enumerate(cal):
    bound=cal[i-2] if i>=2 else pd.NaT
    j=rates.index.searchsorted(bound,side='right')-1 if pd.notna(bound) else -1
    records.append({'qlib_session':day,'source_date':str(rates.index[j].date()) if j>=0 else None,'ten_year_real_yield_percent_lag2':float(rates.iloc[j]) if j>=0 else float('nan')})
all_map=pd.DataFrame(records)
train=all_map[(all_map.qlib_session>=pd.Timestamp('2018-10-15'))&(all_map.qlib_session<=pd.Timestamp('2022-12-30'))]
held=all_map[(all_map.qlib_session>=pd.Timestamp('2023-01-03'))&(all_map.qlib_session<=pd.Timestamp('2026-10-07'))]
assert len(train)==1061 and len(held)==944 and int(train.ten_year_real_yield_percent_lag2.notna().sum())==1060 and held.ten_year_real_yield_percent_lag2.notna().all()
assert train.loc[train.ten_year_real_yield_percent_lag2.isna(),'qlib_session'].dt.strftime('%Y-%m-%d').tolist()==['2018-10-15']
assert str(train.qlib_session.iloc[0].date())=='2018-10-15' and str(held.qlib_session.iloc[-1].date())=='2026-10-07'
map_frame=pd.concat([train,held],ignore_index=True)
coverage={'train_sessions':len(train),'train_known':int(train.ten_year_real_yield_percent_lag2.notna().sum()),'held_sessions':len(held),'held_known':int(held.ten_year_real_yield_percent_lag2.notna().sum()),'first_two_source_dates':train.source_date.iloc[:2].tolist(),'train_unknown_dates':['2018-10-15'],'first_public_and_revision_qualified':False,'source_rows':len(rates)}
with R.start(experiment_name=experiment,uri=str(card/'qlib_recorder')):
    rec=R.get_recorder();rec.save_objects(artifact_path='inputs',repair_plan=repair);rec.save_objects(artifact_path='analysis',source_map=map_frame,coverage=coverage);rec.log_metrics(train_known=coverage['train_known'],held_known=coverage['held_known'],new_model_fits=0,new_native_accounts=0);rid=rec.id;eid=str(rec.experiment_id)
copy_binding=card/'nas_binding_map_repair_v2.json';copy_binding.write_text(json.dumps(binding,indent=2)+'\n')
evidence=archive_nas_recorder(copy_binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='treasury-map-repair-cold-') as temp:
    materialize_nas_evidence(copy_binding,card=Path(temp))
    relative=Path('qlib_recorder')/eid/rid/'artifacts/analysis/source_map'
    assert sha(card/relative)==sha(Path(temp)/relative)
result={'decision':'corrected_Stage1_10y_real_yield_source_map_one_prefix_unknown_PIT_unqualified','parent_recorder_id':repair['parent_recorder_id'],'recorder_id':rid,'experiment_id':eid,'coverage':coverage,'NAS_recorder_sha256':evidence['sha256'],'new_external_GETs':0,'SOXX_outcome_reads':0,'new_model_fits':0,'new_native_accounts':0,'qualified_forward_days':0}
(card/'map_repair_v2_result.json').write_text(json.dumps(result,indent=2)+'\n')
state.update(status='completed',recorder_id=rid);(card/'map_repair_v2_state.json').write_text(json.dumps(state,indent=2)+'\n')
print(json.dumps(result))


{"decision": "corrected_Stage1_10y_real_yield_source_map_one_prefix_unknown_PIT_unqualified", "parent_recorder_id": "4d4fefd49efa4a67a3da0db8d8d378b6", "recorder_id": "3139aaaaad954c37a938b9f23677e41a", "experiment_id": "813340755898088674", "coverage": {"train_sessions": 1061, "train_known": 1060, "held_sessions": 944, "held_known": 944, "first_two_source_dates": [null, "2018-10-12"], "train_unknown_dates": ["2018-10-15"], "first_public_and_revision_qualified": false, "source_rows": 5956}, "NAS_recorder_sha256": "cb3fbbd6ee5f04273d6164efb0f8970a9a6687bfd8a64827740fe78e0bb0bf94", "new_external_GETs": 0, "SOXX_outcome_reads": 0, "new_model_fits": 0, "new_native_accounts": 0, "qualified_forward_days": 0}


# 美国财政部 10 年期实际收益率：Stage1 来源预检

## 中文结论

按冻结计划一次读取美国财政部四份官方 CSV，覆盖 2003-01-02 至 2026-10-22，共 5,956 个唯一来源日期。原始响应、头部、接收时间和 SHA-256 已存入 Yanxuan NAS；远端校验及定向冷恢复通过。财政部说明该利率来自收盘指示性国债报价，因此同日 09:46 开盘决策不能使用当日数值。[官方历史档案](https://home.treasury.gov/resource-center/data-chart-center/interest-rates/daily-treasury-rate-archives)与[口径说明](https://home.treasury.gov/policy-issues/financing-the-government/interest-rate-statistics/interest-rates-frequently-asked-questions)是来源依据。

只用冻结 Stage1 配套的原生 Qlib 交易日历，在决策日之前至少隔两个 Qlib 会话取最近的来源日期。修正后的来源映射在 2018–2022 原训练期 **1060/1061** 日有数，2023–2026 复用评价期 **944/944** 日有数。唯一未知训练日是 **2018-10-15**：配套 provider 最早只到 2018-10-12，不能在同一 Qlib 时钟下证明已有两个前序会话。首次来源映射因裁掉了日历前缀误报 1059/1061；首次修复又错误要求 1061/1061，在新 Recorder 写入前停下。两次过程均保留，第二次机械修复才得到上述保守覆盖。

这条来源可能补充 Stage1 原有 39 项价格、VIX 和广度输入，但**历史首次公开和修订版本尚未核证**。2003–2023 合并 CSV 的原始 HTTP 头显示 `Last-Modified: Wed, 14 Jan 2026 23:01:12 GMT`，可知今天收到的整份文件不是 2018–2022 逐日首次发布的原字节；此头本身不能证明某个数值被修订。两会话延迟只是保守研究假设，不能把回溯 CSV 自动变成逐日当时版本。见 `vintage_gate.json`。尚未定义收益率冲击坐标或做与旧输入的冗余检验，更未读取 SOXX 结果、拟合模型或运行 0bp 账户。下一步应先核对原始版本和决策时钟，再冻结单一坐标、接收层与信息门；只有合格候选才从完整 Stage1 f696+b472 链进入 0bp 加 Tiered 手续费的完整资金账户及 SOXX 硬门。

来源映射 Qlib Recorder `3139aaaaad954c37a938b9f23677e41a` 与 NAS 归档 SHA-256 `cb3fbbd6ee5f04273d6164efb0f8970a9a6687bfd8a64827740fe78e0bb0bf94` 已定向恢复。原始首次 Recorder `4d4fefd49efa4a67a3da0db8d8d378b6` 和错误映射仍保留。SOXX 超额、2/5bp 和独立前瞻证据均未通过；Goal 继续。
